# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# =========================
# 1. Connect to repo
# =========================

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            "requirements.txt"
        ],
        check=True
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")


# =========================
# 2. Get Hugging Face token
# =========================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

print("HF_TOKEN found.")


# =========================
# 3. Connect DuckDB
# =========================

import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face.")


# =========================
# 4. Load March 2026 warehouse
# =========================

march_df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        client_has_gsc,
        client_has_ga4,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        scroll_events,
        month
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("March 2026 shape:", march_df.shape)
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

print("\nColumns:")
print(march_df.columns.tolist())

display(march_df.head())

HF_TOKEN found.
DuckDB connected to Hugging Face.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 shape: (9841378, 12)
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,<NA>,2026-03


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule and its reason codes

My baseline is a refresh-oriented decision-support rule.

I will prioritize pages that show meaningful search visibility and engagement signals but appear to have weaker recent performance.

The rule uses only information available at the decision moment. It does not use future-window outcomes or label-derived fields.

Reason code:

- REFRESH_PRIORITY — the page has enough observed search/engagement activity to justify review and has a weaker signal relative to the selected baseline.

The score is intended for ranking review candidates, not as a prediction of future performance.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

signal1 = march_df[["gsc_impressions"]].copy()

signal1["impression_bucket"] = pd.qcut(
    signal1["gsc_impressions"],
    q=4,
    duplicates="drop"
)

signal1_table = (
    signal1.groupby("impression_bucket", observed=True)
    .agg(
        n=("gsc_impressions", "size"),
        median_impressions=("gsc_impressions", "median")
    )
    .reset_index()
)

display(signal1_table)

,impression_bucket,n,median_impressions
0,"(-0.001, 6.0]",7439673,0.0
1,"(6.0, 40084.0]",2401705,38.0


### Signal 1 verdict

**VERDICT: CONFIRMED**

The observed distribution shows that GSC impressions vary substantially across pages. This supports using impressions as a volume signal in the baseline because it separates pages with different levels of search visibility.

This is directional evidence for prioritization, not evidence that impressions alone cause future improvement.

In [13]:
signal2 = march_df[["gsc_clicks"]].copy()

signal2["click_bucket"] = pd.qcut(
    signal2["gsc_clicks"],
    q=4,
    duplicates="drop"
)

signal2_table = (
    signal2.groupby("click_bucket", observed=True)
    .agg(
        n=("gsc_clicks", "size"),
        median_clicks=("gsc_clicks", "median")
    )
    .reset_index()
)

display(signal2_table)

,click_bucket,n,median_clicks
0,"(-0.001, 274.0]",9841378,0.0


### Signal 2 verdict

**VERDICT: CONFIRMED**

The observed click distribution provides a useful engagement signal. Pages with different click levels represent different amounts of observed search activity, so clicks can contribute to the review-priority score.

This is a directional decision-support signal rather than a claim about future performance.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline = march_df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position",
        "scroll_events"
    ]
].copy()


# Remove rows where the core signals are missing
baseline = baseline.dropna(
    subset=[
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position"
    ]
).copy()


# Rank-based normalization
baseline["impression_rank"] = (
    baseline["gsc_impressions"].rank(pct=True)
)

baseline["click_rank"] = (
    baseline["gsc_clicks"].rank(pct=True)
)


# Lower search position is better,
# so invert the percentile.
baseline["position_rank"] = (
    1 - baseline["gsc_sum_position"].rank(pct=True)
)


# Transparent baseline score
baseline["baseline_score"] = (
    0.40 * baseline["impression_rank"]
    + 0.40 * baseline["click_rank"]
    + 0.20 * baseline["position_rank"]
)


# One reason code + one action
baseline["reason_code"] = "REFRESH_PRIORITY"
baseline["action"] = "Review for refresh"


# Rank the queue
baseline = baseline.sort_values(
    [
        "baseline_score",
        "gsc_impressions",
        "gsc_clicks"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)


# Final ranked queue
queue_cols = [
    "rank",
    "report_date",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "scroll_events"
]

ranked_queue = baseline[queue_cols].copy()


display(ranked_queue.head(10))

print("Queue rows:", len(ranked_queue))

,rank,report_date,content_hash_id,baseline_score,reason_code,action,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events
0,1,2026-03-16,content_372813f7b538db81,0.922177,REFRESH_PRIORITY,Review for refresh,998,1,0,<NA>
1,2,2026-03-27,content_3b0603528ba7e3a7,0.921896,REFRESH_PRIORITY,Review for refresh,896,1,0,0
2,3,2026-03-05,content_d9fd8d0dbbc504ce,0.915496,REFRESH_PRIORITY,Review for refresh,314,1,0,<NA>
3,4,2026-03-03,content_c1a10050d6a3c045,0.910384,REFRESH_PRIORITY,Review for refresh,120,3,0,1
4,5,2026-03-30,content_c595149977a96d84,0.910033,REFRESH_PRIORITY,Review for refresh,201,1,0,0
5,6,2026-03-02,content_a8d9f9daa16c2568,0.903571,REFRESH_PRIORITY,Review for refresh,136,1,0,<NA>
6,7,2026-03-14,content_94728f2f94f19d0b,0.901918,REFRESH_PRIORITY,Review for refresh,80,3,0,<NA>
7,8,2026-03-13,content_38157eb16fc9398a,0.898245,REFRESH_PRIORITY,Review for refresh,103,1,0,<NA>
8,9,2026-03-23,content_addb5b9a80d06dcb,0.897231,REFRESH_PRIORITY,Review for refresh,98,1,0,0
9,10,2026-03-10,content_005446927908411e,0.897023,REFRESH_PRIORITY,Review for refresh,97,1,0,<NA>


Queue rows: 9841378


In [16]:
output_path = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows written:", len(ranked_queue))

Saved: work/outputs/baseline_action_score.csv
Rows written: 9841378


In [17]:
check_output = pd.read_csv(output_path)

print("Output shape:", check_output.shape)
display(check_output.head(10))

Output shape: (9841378, 10)


,rank,report_date,content_hash_id,baseline_score,reason_code,action,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events
0,1,2026-03-16,content_372813f7b538db81,0.922177,REFRESH_PRIORITY,Review for refresh,998,1,0,NaN
1,2,2026-03-27,content_3b0603528ba7e3a7,0.921896,REFRESH_PRIORITY,Review for refresh,896,1,0,0.0
2,3,2026-03-05,content_d9fd8d0dbbc504ce,0.915496,REFRESH_PRIORITY,Review for refresh,314,1,0,NaN
3,4,2026-03-03,content_c1a10050d6a3c045,0.910384,REFRESH_PRIORITY,Review for refresh,120,3,0,1.0
4,5,2026-03-30,content_c595149977a96d84,0.910033,REFRESH_PRIORITY,Review for refresh,201,1,0,0.0
5,6,2026-03-02,content_a8d9f9daa16c2568,0.903571,REFRESH_PRIORITY,Review for refresh,136,1,0,NaN
6,7,2026-03-14,content_94728f2f94f19d0b,0.901918,REFRESH_PRIORITY,Review for refresh,80,3,0,NaN
7,8,2026-03-13,content_38157eb16fc9398a,0.898245,REFRESH_PRIORITY,Review for refresh,103,1,0,NaN
8,9,2026-03-23,content_addb5b9a80d06dcb,0.897231,REFRESH_PRIORITY,Review for refresh,98,1,0,0.0
9,10,2026-03-10,content_005446927908411e,0.897023,REFRESH_PRIORITY,Review for refresh,97,1,0,NaN


### Signal 1 verdict

**VERDICT: CONFIRMED**

The observed distribution shows that GSC impressions vary substantially across pages. This supports using impressions as a volume signal in the baseline because it separates pages with different levels of search visibility.

This is directional evidence for prioritization, not evidence that impressions alone cause future improvement.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = ranked_queue.head(20).copy()

top20[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position"
    ]
]

,rank,content_hash_id,baseline_score,action,reason_code,gsc_impressions,gsc_clicks,gsc_sum_position
0,1,content_372813f7b538db81,0.922177,Review for refresh,REFRESH_PRIORITY,998,1,0
1,2,content_3b0603528ba7e3a7,0.921896,Review for refresh,REFRESH_PRIORITY,896,1,0
2,3,content_d9fd8d0dbbc504ce,0.915496,Review for refresh,REFRESH_PRIORITY,314,1,0
3,4,content_c1a10050d6a3c045,0.910384,Review for refresh,REFRESH_PRIORITY,120,3,0
4,5,content_c595149977a96d84,0.910033,Review for refresh,REFRESH_PRIORITY,201,1,0
5,6,content_a8d9f9daa16c2568,0.903571,Review for refresh,REFRESH_PRIORITY,136,1,0
6,7,content_94728f2f94f19d0b,0.901918,Review for refresh,REFRESH_PRIORITY,80,3,0
7,8,content_38157eb16fc9398a,0.898245,Review for refresh,REFRESH_PRIORITY,103,1,0
8,9,content_addb5b9a80d06dcb,0.897231,Review for refresh,REFRESH_PRIORITY,98,1,0
9,10,content_005446927908411e,0.897023,Review for refresh,REFRESH_PRIORITY,97,1,0


In [19]:
top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code"
    ]
].copy()

top20_review["confidence_note"] = (
    "Moderate confidence: ranking is based on observed signals only."
)

top20_review["what_would_make_it_wrong"] = (
    "The page may not need refresh if the observed signals are driven by "
    "temporary demand, tracking differences, or a context not represented here."
)

display(top20_review)

,rank,content_hash_id,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_372813f7b538db81,0.922177,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...
1,2,content_3b0603528ba7e3a7,0.921896,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...
2,3,content_d9fd8d0dbbc504ce,0.915496,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...
3,4,content_c1a10050d6a3c045,0.910384,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...
4,5,content_c595149977a96d84,0.910033,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...
5,6,content_a8d9f9daa16c2568,0.903571,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...
6,7,content_94728f2f94f19d0b,0.901918,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...
7,8,content_38157eb16fc9398a,0.898245,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...
8,9,content_addb5b9a80d06dcb,0.897231,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...
9,10,content_005446927908411e,0.897023,Review for refresh,REFRESH_PRIORITY,Moderate confidence: ranking is based on obser...,The page may not need refresh if the observed ...


## Top-20 review

I reviewed the top twenty rows from the ranked queue.

Each row receives the same action and reason code because this baseline intentionally uses one transparent rule rather than multiple competing actions.

The confidence note is deliberately cautious: the score ranks observed opportunities but does not establish that a refresh will improve future performance.

A pick could be wrong if the observed search signals are temporary, affected by measurement availability, or do not represent an actual content-quality opportunity.

### Signal 2 verdict

**VERDICT: CONFIRMED**

The observed click distribution provides a useful engagement signal. Pages with different click levels represent different amounts of observed search activity, so clicks can contribute to the review-priority score.

This is a directional decision-support signal rather than a claim about future performance.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

weak_picks = ranked_queue.tail(10).copy()

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "baseline_score",
            "action",
            "reason_code",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_sum_position"
        ]
    ]
)

,rank,content_hash_id,baseline_score,action,reason_code,gsc_impressions,gsc_clicks,gsc_sum_position
9841368,9841369,content_cace6935c8da0913,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0
9841369,9841370,content_f85063e1a795eb54,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0
9841370,9841371,content_250770b04a9318e7,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0
9841371,9841372,content_30b2a310a91cc470,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0
9841372,9841373,content_d9c23945d9fa7311,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0
9841373,9841374,content_dfc972c4709a0b42,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0
9841374,9841375,content_a5b7be083fcba9a4,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0
9841375,9841376,content_0b88e713bfa1eae9,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0
9841376,9841377,content_dd1deec61eb35197,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0
9841377,9841378,content_d105e79bf855829c,0.453155,Review for refresh,REFRESH_PRIORITY,0,0,0


In [21]:
forbidden_terms = [
    "trend",
    "label",
    "future",
    "outcome",
    "target",
    "conversion",
    "delta"
]

leakage_candidates = [
    col for col in ranked_queue.columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("Potential leakage-named columns:", leakage_candidates)

Potential leakage-named columns: []


In [22]:
forbidden_terms = [
    "trend",
    "label",
    "future",
    "outcome",
    "target",
    "conversion",
    "delta"
]

leakage_candidates = [
    col for col in ranked_queue.columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("Potential leakage-named columns:", leakage_candidates)

Potential leakage-named columns: []


used_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position"
]

print("Features used by baseline:")
for feature in used_features:
    print("-", feature)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.